# homology-audit tutorial — audit your DTA/PPI benchmark in five steps

This notebook walks through the complete **homology-audit v1.2.0** toolkit on a
tiny synthetic dataset that ships with the package:

1. **Prepare** — normalize any common benchmark format (split folder, long CSV,
   BindingDB export, FASTA pair, DAVIS/KIBA affinity matrix) into
   `<name>_train.csv` / `<name>_test.csv`.
2. **Audit** — MMseqs2 clustering at 40/60/80% identity; every test target
   gets its nearest training identity and a same-cluster/cross-cluster label.
3. **Mirror** — binding-spectrum (ΔΔG) redundancy: even sequence-remote test
   targets may repeat a training target's mutational response profile.
4. **Evaluate** — recompute model metrics on the standard vs the audited
   (homology-free) panel; detect winner reversals and mis-selection costs.
5. **Degradation** — automatic severity verdicts (OK / WARN / FAIL).

Everything shown here can also be run from the shell as a single command
(see the last section), or via `homology-audit <command>` subcommands.

**Prerequisite:** MMseqs2 must be installed (`conda install -c conda-forge mmseqs2`,
`brew install mmseqs2`, or a release binary). The path can be given via the
`HOMOLOGY_AUDIT_MMSEQS` environment variable.

In [ ]:
import os
from pathlib import Path

import pandas as pd

# If mmseqs is not on your PATH, point the toolkit at the binary:
# os.environ["HOMOLOGY_AUDIT_MMSEQS"] = "/path/to/mmseqs"   # or mmseqs.exe

import homology_audit
from homology_audit.audit import run_audit
from homology_audit.degradation import run_degradation
from homology_audit.evaluate import run_evaluation
from homology_audit.mirror import run_mirror
from homology_audit.pipeline import run_audit_benchmark
from homology_audit.adapters import prepare_inputs

print("homology-audit", homology_audit.__version__)
ROOT = Path(homology_audit.__file__).resolve().parents[1]
OUT = ROOT / "examples" / "output_tutorial"
OUT.mkdir(exist_ok=True)

## 0. Generate the synthetic demo data

Two benchmark panels (`demo`, `demo2`) with 160-residue protein sequences:
some test targets are homologous to training targets (same protein family),
others are fully remote. Predictions from three fake models are synthesized so
that the *standard-panel* winner differs from the *audited-panel* winner.
A mutation-level ΔΔG panel (`demo_mutation_ddg.csv`) is also generated for the
mirror audit.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(ROOT / "examples" / "make_example_data.py")],
               check=True)
data_dir = ROOT / "examples" / "data"
sorted(p.name for p in data_dir.glob("*"))

## 1. Prepare — benchmark format adapters

`prepare_inputs` accepts:

| layout | example |
|---|---|
| split directory | `data/` with `demo_train.csv` + `demo_test.csv` |
| single long CSV with a split column | preprocessed Davis/KIBA releases, BindingDB TSV/CSV exports |
| single long CSV without split | give `test_data=` or `test_targets=` |
| FASTA pair | `train.fasta` + `test.fasta` |
| DAVIS/KIBA affinity matrix | `affinity.mat` + `--targets-file` + `--sequences-fasta` (+ optional `test_targets`) |

The sequence column is auto-detected (`target_sequence`,
`BindingDB_Target_Chain_Sequence`, `target_seq`, `Sequence`, ...) and always
normalized to `target_sequence`.

In [ ]:
prep = prepare_inputs(data=data_dir, out_dir=OUT / "prepared", name="demo")
print("format:", prep["format"])
print("train:", prep["train_csv"], "| test:", prep["test_csv"])

## 2. Audit — MMseqs2 clustering

`run_audit` clusters the merged train+test sequences at every identity
threshold and reports, per test target, the nearest training identity and a
`same_cluster_<tag>` label. The key number is the **cross-cluster ratio** —
the fraction of test targets that share no cluster with any training target
(the homology-free fraction).

In [ ]:
audit = run_audit(name="demo", out_dir=OUT / "audit",
                  train_csv=prep["train_csv"], test_csv=prep["test_csv"])
pd.DataFrame(audit["summary_rows"])

In [ ]:
per_target = pd.read_csv(OUT / "audit" / "demo" / "per_target_audit.csv")
per_target.head()

## 3. Mirror — binding-spectrum redundancy

A test complex can be sequence-remote yet still *mirror* a training complex:
the same mutations produce the same ΔΔG ordering (shared-mutation Spearman ρ)
or the ΔΔG distributions coincide (KS similarity = 1 − KS statistic).
Each test complex gets its maximum score over all training complexes;
scores ≥ 0.7 flag hidden redundancy that sequence clustering cannot see.

Input: a long CSV of mutation-level measurements —
`complex_id, split, mutation, ddG` (or affinity columns from which ΔΔG is
derived). Passing `per_target_audit` restricts the headline numbers to
sequence-cross-cluster complexes.

In [ ]:
mirror = run_mirror(
    data_csv=data_dir / "demo_mutation_ddg.csv",
    out_dir=OUT / "mirror", dataset_name="demo",
    per_target_audit=OUT / "audit" / "demo" / "per_target_audit.csv",
    target_id_col="target_id")
mirror["summary"]

In [ ]:
pc = mirror["per_complex"]
pc[["test_complex", "max_spearman_rho", "best_train_spearman",
    "max_ks_similarity", "best_train_ks", "cross_cluster"]]

## 4. Evaluate — standard vs audited panel

Predictions are a long CSV: `dataset, model, target_sequence, y_true, y_pred`.
Every model's metrics are recomputed on (a) the full standard test panel and
(b) the audited homology-free subset, then rankings are compared.

In [ ]:
ev = run_evaluation(
    predictions_csv=ROOT / "examples" / "predictions" / "demo_predictions.csv",
    labels_root=OUT / "audit", datasets=["demo"], out_dir=OUT / "evaluation",
    metrics=["R2", "EF@1%", "ECE"])
ev["ranking"]

## 5. Degradation — automatic severity verdicts

`run_degradation` consumes the evaluation outputs (or raw predictions) and
appends `rank_flips`, `n_degraded_models` and a `severity` column:

* **FAIL** — winner reversed *and* relative mis-selection cost ≥ 10%
* **WARN** — winner reversed *or* cost ≥ 5%
* **OK** — otherwise

In [ ]:
deg = run_degradation(out_dir=OUT / "degradation",
                      ranking_csv=OUT / "evaluation" / "rank_comparison.csv",
                      per_model_csv=OUT / "evaluation" / "audited_evaluation_per_model.csv")
deg["report"][["dataset", "metric", "winner_reversed",
               "relative_cost_pct", "rank_flips", "severity"]]

## All of the above in one command

The `audit-benchmark` entry point chains prepare → audit → mirror → evaluate →
degradation and writes `audit-benchmark_summary.txt`:

```bash
audit-benchmark --data examples/data --output my_output --name demo \
    --predictions examples/predictions/demo_predictions.csv
```

Every parameter can alternatively live in a YAML config
(`configs/default_audit.yaml` is the annotated template):

```bash
audit-benchmark --output my_output --config configs/demo.yaml
```

In [ ]:
res = run_audit_benchmark(
    data=data_dir, output=OUT / "pipeline", config_path=None,
    overrides={"dataset": {"name": "demo", "data": str(data_dir)},
               "evaluation": {"predictions":
                              str(ROOT / "examples" / "predictions" /
                                  "demo_predictions.csv")}})
print((OUT / "pipeline" / "audit-benchmark_summary.txt").read_text())

## Output layout

```
my_output/
├── prepared/                     # normalized <name>_{train,test}.csv
├── audit/<name>/                 # sequences_*.fasta, clu_*/,
│   ├── per_target_audit.csv      #   nearest identity + same_cluster_<tag>
│   └── audit_summary_dataset.csv #   cross-cluster ratios per threshold
├── mirror/                       # per_complex_mirror.csv, mirror_summary.csv
├── evaluation/                   # rank_comparison.csv, per-model table
├── degradation/                  # degradation_report.{csv,txt} (severity)
└── audit-benchmark_summary.txt   # one-page overview
```

### Next steps with real data

* **Davis / KIBA**: point `--data` at the preprocessed CSV (needs a sequence
  column + split column) or at the raw `affinity.mat` with
  `--targets-file`/`--sequences-fasta`.
* **BindingDB**: export any target-family subset; the adapter picks up
  `BindingDB_Target_Chain_Sequence` automatically.
* **SKEMPI v2 / PPI**: build the mutation-level ΔΔG table (`ddG = R·T·ln(Kd_wt/Kd_mut)`)
  and run the `mirror` command to quantify functional redundancy.
* Robustness extras: `router`, `sensitivity`, `algorithm-check`,
  `negative-control`, `band-lodo` — see the README and
  `homology-audit <command> --help`.